# Book Writer with Agents: practice

This module/project pack is extracted from the local files listed below. Functions and line numbers are source locations; the original program is not executed by practice.

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [2_openai/community_contributions/Book Writer with Agents/dump_notebook_cells.py](../../../../../../2_openai/community_contributions/Book%20Writer%20with%20Agents/dump_notebook_cells.py)
- [2_openai/community_contributions/Book Writer with Agents/README.md](../../../../../../2_openai/community_contributions/Book%20Writer%20with%20Agents/README.md)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. Configuration stays outside executable examples

Environment-variable names identify settings. Values belong in .environment/.env and are loaded deliberately. This exercise inspects only the path and simulated presence flags; it reads no private file or variable values.

**Predict:** Does finding a file prove that authentication works?

In [ ]:
from pathlib import Path
# Find the repository from a kernel started inside it.
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "1_foundations" / "3_lab3.ipynb").is_file()), None)
if root is None:
    raise FileNotFoundError("Start your kernel inside ai_agents")
configuration_path = root / ".environment" / ".env"
print("Configuration path:", configuration_path)
simulated_presence = {"OPENAI_API_KEY": True, "GOOGLE_API_KEY": False}
print(simulated_presence)  # Booleans only; these are fictional flags.
assert configuration_path.name == ".env"

**Check:** No. File existence, variable presence, and successful provider authentication are separate checks.

**Change one thing:** Explain the difference between a missing file, a missing variable, and a provider rejecting authentication. Do not print token prefixes or values.

## 2. Structured output validates shape

Pydantic models name the fields you expect and check constraints. This helps callers use results reliably, but a valid field can still contain a false claim.

**Predict:** Will confidence=2 pass the constraint?

In [ ]:
from pydantic import BaseModel, Field, ValidationError
class Report(BaseModel):
    summary: str
    confidence: float = Field(ge=0, le=1)
report = Report(summary="Fictional result", confidence=0.8)
print(report.model_dump())
try:
    Report(summary="Bad shape", confidence=2)
except ValidationError:
    print("Invalid confidence rejected")
else:
    raise AssertionError("Constraint was not enforced")

**Check:** No. The value must be between zero and one. Validation checks the constraint, not the truth of the summary.

**Change one thing:** Add a field named follow_up_questions with type list[str]. Construct a valid object using it.

## 3. A guardrail makes a stop/continue decision

An input guardrail checks a request before later work. The tripwire is a boolean. This deterministic example teaches the branch; it is not the model-based name detector in the source.

**Predict:** Does Alice trigger this demonstration guardrail?

In [ ]:
from pydantic import BaseModel
class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name: str
def check_name(message):
    found = "Alice" in message  # Deliberately limited teaching rule.
    return NameCheckOutput(is_name_in_message=found, name="Alice" if found else "")
for request in ["Send from Alice", "Send from Head of Business Development"]:
    result = check_name(request)
    tripwire_triggered = result.is_name_in_message
    print("stop" if tripwire_triggered else "continue")
assert check_name("Send from Alice").is_name_in_message

**Check:** Yes for Alice, no for the title. A boolean decision does not establish that every personal name is detected.

**Change one thing:** Try another name and observe the limitation. Describe how you would evaluate false positives and missed names.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.